In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Tools

In [ ]:
from langchain.tools import tool
country_capitals = {
    "France": "Paris",
    "Japan": "Tokyo",
    "Canada": "Ottawa",
    "Germany": "Berlin",
    "Australia": "Canberra",
    "Egypt": "Cairo",
    "Mexico": "Mexico City",
    "South Africa": "Pretoria",
}


@tool
def get_capital(country: str) -> dict:
    """Return structured capital information for a given country."""
    normalized = " ".join(word.capitalize() for word in country.strip().split())

    capital = country_capitals.get(normalized)

    if capital:
        return {
            "country": normalized,
            "found": True,
            "capital": capital,
            "fallback_allowed": False,
        }

    return {
        "country": normalized,
        "found": False,
        "capital": None,
        "fallback_allowed": normalized == "India",
    }

# Model
- model are the reasoning engine of agents
- models can be utilized in 2 ways:
    - With agent: you can specify hardcode "provider:model"
    - standalone: create a model object and invoke prompt with it


In [ ]:
#gemini Model
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "gemini-3.1-flash-lite",
    model_provider="google_genai",
    temperature=0.2,
    timeout=600,
    max_tokens=1000,
)
#model.bind_tools([get_capital]): binding model with tool


In [ ]:
#checking standalone model
result=model.invoke("how much 1+1")
print(result['messages'][-1].content[0]['text'])

In [ ]:
#nvidia model
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(
  model="deepseek-ai/deepseek-v4-flash",
  api_key=os.getenv("NVIDIA_API_KEY"),
  temperature=0,
  top_p=0.95,
  max_tokens=1000,
)

# Memory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

# Structured Output

In [ ]:
from typing import Dict, Literal, Optional
from pydantic import BaseModel, Field, RootModel


class CapitalInfo(BaseModel):
    country: Optional[str] = Field(default=None)
    capital: Optional[str] = Field(default=None)
    source: Optional[Literal["tool", "AI fallback", "tool-not-found"]] = Field(default=None)
    explanation: Optional[str] = Field(default=None)


class Response(BaseModel):
    result: Dict[str, CapitalInfo] = Field(
        description="Dictionary where key is question number as string and value is the answer object"
    )

# MiddleWare

In [ ]:
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware

subagents_list=[
    {
        "name":"model1",
        "system_prompt":"",
    },
    {
        "name":"model2",
        "system_prompt":"",
    }
]

middleware_list=[
    ModelRetryMiddleware(max_retries=3),
    ToolRetryMiddleware(max_retries=2),
    SubAgentMiddleware(
        subagents=subagents_list
    )
]

# Message

- Message Represent input and output of model
- Text Prompt: straight forward Raw string
    - single, standalone request
    - dont persist conversation history
- Message Prompt
    - Usages
        - Multi-turn , multi model conversations
    - Message Contains
        - Role: Identify the message type (system or user)
        - Content: Represent the actual content of Message (text, image, audio)
        - Metadata: Optional metadata
    - Message Type
        - System Message: Define model's behaviour
        - Human Message: User interaction or input to model
        - AI Message: Responses generated by the model
        - Tool Message: Reprsent the output of tool calls
    

In [ ]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]
response = model.invoke(messages)


# Agent
- agent=model+harness (skills, tools, contest, subagent, system prompt, memory)
- it is model calling tools in a loop until a given task is complete

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

SYSTEM_PROMPT = """
You are a helpful assistant.

Rules:
- For each question, provide a structured response according to the Response schema.
- Only call `get_capital` when the user asks directly for a country's capital.
- If the question is about what to do, rules, or process, do NOT call `get_capital`.
- For each capital question, call `get_capital` at most one time.
- After `get_capital` returns data for a question, DO NOT call the tool again for that same question.
- Use the returned tool data to produce the final structured response.

Decision rules for capital questions:
- If found=True:
  country = tool country
  capital = tool capital
  source = "tool"
  explanation = null

- If found=False and fallback_allowed=True:
  country = tool country
  capital = your own knowledge
  source = "AI fallback"
  explanation = null

- If found=False and fallback_allowed=False:
  country = tool country
  capital = "Capital not found in tool"
  source = "tool-not-found"
  explanation = null

For non-capital/process questions:
- Do not call the tool.
- country = null
- capital = null
- source = null
- explanation = answer to the process question

When all questions are handled, return the final structured response.
"""

agent = create_agent(
    name="CapitalFinder",
    model=model, #you can also pass non-standalone i.e. "provider:model" directly
    tools=[get_capital],
    system_prompt=SYSTEM_PROMPT,
    #response_format=Response,
    middleware=middleware_list,
    
)

# Invocation

- it can be done in 3 ways
    - invoke
    - stream
    - batch

## invoke

A follow-up turn on the same conversation: reuse the same thread_id to keep history

In [ ]:
from langchain_core.utils.uuid import uuid7

content = """
Answer each question with both the capital and the source.

1) What is the capital of India?
2) What is the capital of Brazil?
3) What is the capital of France?
4) What should you do if asked about India?
"""

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the capital of japan?"}]},
    config=config1,
)
print(result['messages'][-1].content[0]['text'])

# A follow-up turn on the same conversation: reuse the same thread_id to keep history
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what about india?"}]},
    config=config1,
)
print(result['messages'][-1].content[0]['text'])


Different conversation: different thread_id history not saved

In [ ]:
from langchain_core.utils.uuid import uuid7
config2 = {"configurable": {"thread_id": str(uuid7())}}
# Different conversation: different thread_id history not saved
result = agent.invoke(
    {"messages": [{"role": "user", "content": "find for france?"}]},
    config=config2,
)
print(result3['messages'][-1].content[0]['text'])


# Streaming

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

content = """
1) What is the capital of India?
2) What is the capital of Brazil?
3) What is the capital of France?
4) What should you do if asked about India?
"""

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": content}]},
    version="v3",
    config={
        "recursion_limit": 8,
        "configurable": {
            "thread_id": "capital-stream-debug-1"
        }
    },
)

for snapshot in stream.values:
    latest_message = snapshot["messages"][-1]

    if isinstance(latest_message, HumanMessage):
        print(f"\nUSER:\n{latest_message.content}")

    elif isinstance(latest_message, AIMessage):
        # Case 1: Agent wants to call a tool
        if latest_message.tool_calls:
            print("\nAGENT IS CALLING TOOL:")
            for tc in latest_message.tool_calls:
                print(f"Tool name: {tc['name']}")
                print(f"Tool args: {tc['args']}")

        # Case 2: Agent gives final answer
        elif latest_message.content:
            print("\nFINAL AGENT ANSWER:")
            print(latest_message.content)

    elif isinstance(latest_message, ToolMessage):
        print("\nTOOL RESULT:")
        print(f"Tool name: {latest_message.name}")
        print(f"Tool output: {latest_message.content}")

## Batch

In [ ]:
## Not Tested Yet

questions = [
    "What is the capital of India?",
    "What is the capital of Brazil?",
    "What is the capital of France?",
    "What should you do if asked about India?",
]

inputs = [{"messages": [{"role": "user", "content": q}]} for q in questions]

responses = agent.batch(inputs, config={"recursion_limit": 20})

for i, res in enumerate(responses, 1):
    print(f"\n--- Response {i} ---")
    print(res["messages"][-1].content)